# 🤖 AutoML Pilot - Pro Training Template
Use this Colab notebook to run advanced AutoML (PyCaret) on your dataset with automated reporting and EDA.

In [ ]:
# @title 1. Install Dependencies
!pip install -q pycaret[full] ydata-profiling pandas jinja2 shap
print('✅ Dependencies installed!')

In [ ]:
# @title 2. Load Dataset
import pandas as pd
import os
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(f'✅ Loaded {filename}: {df.shape[0]} rows')

In [ ]:
# @title 3. Automated EDA
run_eda = True # @param {type:"boolean"}
if run_eda:
    from ydata_profiling import ProfileReport
    profile = ProfileReport(df, title="Profiling Report", explorative=True)
    profile.to_notebook_iframe()

In [ ]:
# @title 4. AutoML Setup & Training
from pycaret.classification import setup as cls_setup, compare_models as cls_compare, pull as cls_pull, save_model as cls_save, finalize_model as cls_finalize, interpret_model as cls_interpret
from pycaret.regression import setup as reg_setup, compare_models as reg_compare, pull as reg_pull, save_model as reg_save, finalize_model as reg_finalize, interpret_model as reg_interpret
import json

target_column = 'target' # @param {type:"string"}
task_type = 'classification' # @param ["classification", "regression"]

if task_type == 'classification':
    s = cls_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = cls_compare()
    leaderboard = cls_pull()
    final_model = cls_finalize(best_model)
    cls_save(final_model, 'best_model')
    # try: cls_interpret(final_model)
    # except: pass
else:
    s = reg_setup(data=df, target=target_column, session_id=123, verbose=False)
    best_model = reg_compare()
    leaderboard = reg_pull()
    final_model = reg_finalize(best_model)
    reg_save(final_model, 'best_model')
    # try: reg_interpret(final_model)
    # except: pass

# Create metadata for restoration
metadata = {
    "target": target_column,
    "task": task_type,
    "features": [c for c in df.columns if c != target_column],
    "metrics": leaderboard.iloc[0].to_dict()
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f)

display(leaderboard.head())

In [ ]:
# @title 5. Automated Email Reporting
import smtplib, ssl
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

recipient_email = '' # @param {type:"string"}
sender_email = '' # @param {type:"string"}
sender_password = '' # @param {type:"string"}

if recipient_email and sender_email and sender_password:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = f"AutoML Pilot Pro - {task_type.capitalize()} Report"
    msg["From"] = sender_email
    msg["To"] = recipient_email

    html_template = f"""
    <html>
    <body style='font-family: Arial, sans-serif;'>
        <h2 style='color: #4CAF50;'>🚀 AutoML Training Report</h2>
        <p>Your model has been trained successfully using <b>AutoML Pilot Pro</b>.</p>
        <h3>📊 Top Models Leaderboard</h3>
        <div style='overflow-x:auto;'>
            {leaderboard.head().to_html(classes='table table-striped')}
        </div>
        <br>
        <p style='font-size: 0.8em; color: #777;'>Generated via AutoML Pilot Pro Colab Integration</p>
    </body>
    </html>
    """
    msg.attach(MIMEText(html_template, "html"))
    context = ssl.create_default_context()
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
            server.login(sender_email, sender_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print('✅ Email report sent!')
    except Exception as e:
        print(f'❌ Failed to send email: {e}')

In [ ]:
# @title 6. Download Model & Metadata
from google.colab import files
if os.path.exists('best_model.pkl'):
    files.download('best_model.pkl')
if os.path.exists('model_metadata.json'):
    files.download('model_metadata.json')
print('✅ Files ready for download!')